# Finly — Fase 3: Análise de Padrões Comportamentais

Neste notebook vamos transformar o extrato categorizado em **insights reais** sobre o comportamento financeiro.

Perguntas que vamos responder:
- Em quais dias da semana você gasta mais?
- Qual semana do mês pesa mais no bolso?
- Como os gastos variam ao longo do ano?
- Quais categorias têm padrões sazonais?
- Existe algum gasto anômalo — aquele mês que fugiu do normal?

## 1. Imports e carregamento

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Carrega o extrato já categorizado
df = pd.read_csv("../extrato_categorizado.csv", parse_dates=["data"])

# Colunas auxiliares de tempo
df["dia_semana"]    = df["data"].dt.day_name(locale="pt_BR").str.capitalize()
df["mes"]           = df["data"].dt.month
df["mes_nome"]      = df["data"].dt.strftime("%b").str.capitalize()
df["semana_do_mes"] = ((df["data"].dt.day - 1) // 7) + 1
df["ano_mes"]       = df["data"].dt.to_period("M").astype(str)

# Ordem correta dos dias
ORDEM_DIAS = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"]
ORDEM_MESES = ["Jan","Fev","Mar","Abr","Mai","Jun",
               "Jul","Ago","Set","Out","Nov","Dez"]

print(f"✅ {len(df)} transações carregadas")
print(f"Categorias: {sorted(df['categoria'].unique())}")
df.head()

✅ 459 transações carregadas
Categorias: ['alimentação', 'casa', 'delivery', 'educação', 'lazer', 'outros', 'saúde', 'transporte', 'vestuário']


,data,descricao,valor,categoria,dia_semana,mes,mes_nome,semana_do_mes,ano_mes
0,2024-01-01,Hortifruti Verde,419.61,alimentação,Segunda-feira,1,Jan,1,2024-01
1,2024-01-01,iFood - McDonald's,70.35,delivery,Segunda-feira,1,Jan,1,2024-01
2,2024-01-01,iFood - Frango Assado,45.73,delivery,Segunda-feira,1,Jan,1,2024-01
3,2024-01-02,Boate Floresta,42.98,lazer,Terça-feira,1,Jan,1,2024-01
4,2024-01-03,iFood - Frango Assado,50.32,delivery,Quarta-feira,1,Jan,1,2024-01


## 2. Visão geral — gasto por categoria

In [2]:
# Total por categoria
por_categoria = (
    df.groupby("categoria")["valor"]
    .agg(["sum", "count", "mean"])
    .rename(columns={"sum": "total", "count": "qtd", "mean": "ticket_medio"})
    .sort_values("total", ascending=False)
    .reset_index()
)
por_categoria["percentual"] = por_categoria["total"] / por_categoria["total"].sum() * 100

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "bar"}]],
    subplot_titles=("Distribuição do gasto", "Total por categoria (R$)")
)

fig.add_trace(go.Pie(
    labels=por_categoria["categoria"],
    values=por_categoria["total"].round(2),
    hole=0.4,
    textinfo="label+percent"
), row=1, col=1)

fig.add_trace(go.Bar(
    x=por_categoria["categoria"],
    y=por_categoria["total"].round(2),
    text=por_categoria["total"].apply(lambda x: f"R$ {x:,.0f}"),
    textposition="outside",
    marker_color="#636EFA"
), row=1, col=2)

fig.update_layout(title="Gastos anuais por categoria", height=450, showlegend=False)
fig.show()

print("\nResumo:")
print(por_categoria.to_string(index=False))


Resumo:
  categoria    total  qtd  ticket_medio  percentual
alimentação 20038.70  111    180.528829   32.144152
 transporte 18626.40  128    145.518750   29.878677
   delivery  6568.75  103     63.774272   10.536956
      lazer  5938.15   55    107.966364    9.525408
  vestuário  3959.79   15    263.986000    6.351914
     outros  3152.50   19    165.921053    5.056937
      saúde  2297.96   13    176.766154    3.686166
   educação   923.68   11     83.970909    1.481678
       casa   834.18    4    208.545000    1.338111


## 3. Padrão por dia da semana

> **Insight esperado:** delivery deve ser mais caro nas sextas e sábados.

In [3]:
por_dia = (
    df.groupby(["dia_semana", "categoria"])["valor"]
    .mean()
    .reset_index()
    .rename(columns={"valor": "media"})
)

# Filtra categorias mais relevantes
cats_foco = ["delivery", "lazer", "transporte", "alimentação"]
por_dia_foco = por_dia[por_dia["categoria"].isin(cats_foco)]

fig = px.bar(
    por_dia_foco,
    x="dia_semana",
    y="media",
    color="categoria",
    barmode="group",
    category_orders={"dia_semana": ORDEM_DIAS},
    title="Gasto médio por dia da semana",
    labels={"media": "Gasto médio (R$)", "dia_semana": ""},
    height=400
)
fig.show()

# Insight automático
delivery_dia = por_dia[por_dia["categoria"] == "delivery"].set_index("dia_semana")["media"]
if len(delivery_dia) > 0:
    dia_pico = delivery_dia.idxmax()
    dia_min  = delivery_dia.idxmin()
    ratio    = delivery_dia.max() / delivery_dia.min()
    print(f"\n💡 Insight: delivery é {ratio:.1f}x mais caro na {dia_pico} do que na {dia_min}")


💡 Insight: delivery é 2.1x mais caro na Sexta-feira do que na Domingo


## 4. Padrão por semana do mês

> **Insight esperado:** alimentação deve ser maior na 1ª semana (dia de mercadão logo após o salário).

In [4]:
por_semana = (
    df.groupby(["semana_do_mes", "categoria"])["valor"]
    .mean()
    .reset_index()
    .rename(columns={"valor": "media"})
)

fig = px.line(
    por_semana[por_semana["categoria"].isin(cats_foco)],
    x="semana_do_mes",
    y="media",
    color="categoria",
    markers=True,
    title="Gasto médio por semana do mês",
    labels={"media": "Gasto médio (R$)", "semana_do_mes": "Semana do mês"},
    height=400
)
fig.update_xaxes(tickvals=[1,2,3,4], ticktext=["1ª semana","2ª semana","3ª semana","4ª semana"])
fig.show()

# Insight automático
alim_semana = por_semana[por_semana["categoria"] == "alimentação"].set_index("semana_do_mes")["media"]
if len(alim_semana) > 0:
    semana_pico = alim_semana.idxmax()
    print(f"\n💡 Insight: o maior gasto com alimentação ocorre na {semana_pico}ª semana do mês")
    print(f"   Média na {semana_pico}ª semana: R$ {alim_semana.max():.2f}")
    outras = alim_semana.drop(semana_pico).mean()
    print(f"   Média nas demais semanas: R$ {outras:.2f}")


💡 Insight: o maior gasto com alimentação ocorre na 1ª semana do mês
   Média na 1ª semana: R$ 336.66
   Média nas demais semanas: R$ 139.32


## 5. Sazonalidade mensal

> **Insight esperado:** lazer explode em dezembro, vestuário em janeiro e julho.

In [5]:
por_mes = (
    df.groupby(["mes", "mes_nome", "categoria"])["valor"]
    .sum()
    .reset_index()
    .rename(columns={"valor": "total"})
)

fig = px.line(
    por_mes,
    x="mes",
    y="total",
    color="categoria",
    markers=True,
    title="Gasto total por mês e categoria",
    labels={"total": "Total (R$)", "mes": "Mês"},
    height=450
)
fig.update_xaxes(tickvals=list(range(1,13)), ticktext=ORDEM_MESES)
fig.show()

# Heatmap de categorias x meses
pivot = por_mes.pivot(index="categoria", columns="mes", values="total").fillna(0)
pivot.columns = ORDEM_MESES

fig2 = px.imshow(
    pivot,
    title="Heatmap: intensidade de gasto por categoria e mês",
    labels={"color": "Total R$"},
    color_continuous_scale="Blues",
    height=350
)
fig2.show()

## 6. Detecção de anomalias

Meses onde o gasto fugiu do padrão histórico — útil pra identificar gastos extraordinários.

In [6]:
# Gasto total por mês
total_mes = df.groupby("ano_mes")["valor"].sum().reset_index()
total_mes.columns = ["mes", "total"]

# Anomalia = mês com gasto acima de média + 1.5 desvios padrão
media   = total_mes["total"].mean()
desvio  = total_mes["total"].std()
limite  = media + 1.5 * desvio

total_mes["anomalia"] = total_mes["total"] > limite

fig = go.Figure()
fig.add_trace(go.Bar(
    x=total_mes["mes"],
    y=total_mes["total"],
    marker_color=["#EF553B" if a else "#636EFA" for a in total_mes["anomalia"]],
    name="Gasto mensal"
))
fig.add_hline(
    y=limite, line_dash="dash", line_color="orange",
    annotation_text=f"Limite anomalia (R$ {limite:,.0f})"
)
fig.add_hline(
    y=media, line_dash="dot", line_color="green",
    annotation_text=f"Média (R$ {media:,.0f})"
)
fig.update_layout(
    title="Gasto mensal — detecção de anomalias",
    xaxis_title="Mês", yaxis_title="Total (R$)",
    height=400
)
fig.show()

anomalias = total_mes[total_mes["anomalia"]]
if len(anomalias) > 0:
    print("\n💡 Meses anômalos detectados:")
    for _, row in anomalias.iterrows():
        pct = (row['total'] - media) / media * 100
        print(f"   {row['mes']}: R$ {row['total']:,.2f}  ({pct:+.1f}% acima da média)")
else:
    print("\n✅ Nenhum mês anômalo detectado")


💡 Meses anômalos detectados:
   2024-11: R$ 6,736.61  (+29.7% acima da média)
   2024-12: R$ 6,696.92  (+28.9% acima da média)


## 7. Resumo dos insights

In [7]:
gasto_total = df["valor"].sum()
media_mensal = df.groupby("ano_mes")["valor"].sum().mean()
cat_maior = por_categoria.iloc[0]

print("=" * 50)
print("RESUMO — FINLY INSIGHTS")
print("=" * 50)
print(f"Gasto total no ano:      R$ {gasto_total:>10,.2f}")
print(f"Média mensal:            R$ {media_mensal:>10,.2f}")
print(f"Categoria campeã:        {cat_maior['categoria']} ({cat_maior['percentual']:.1f}% do total)")
print()

# Dia mais caro
dia_total = df.groupby("dia_semana")["valor"].mean()
print(f"Dia mais caro:           {dia_total.idxmax()} (média R$ {dia_total.max():.2f})")
print(f"Dia mais barato:         {dia_total.idxmin()} (média R$ {dia_total.min():.2f})")
print()

# Mês mais caro
mes_total = df.groupby("mes_nome")["valor"].sum()
print(f"Mês mais caro:           {mes_total.idxmax()} (R$ {mes_total.max():,.2f})")
print(f"Mês mais barato:         {mes_total.idxmin()} (R$ {mes_total.min():,.2f})")
print("=" * 50)

RESUMO — FINLY INSIGHTS
Gasto total no ano:      R$  62,340.11
Média mensal:            R$   5,195.01
Categoria campeã:        alimentação (32.1% do total)

Dia mais caro:           Terça-feira (média R$ 149.84)
Dia mais barato:         Quinta-feira (média R$ 108.84)

Mês mais caro:           Nov (R$ 6,736.61)
Mês mais barato:         Oct (R$ 4,319.97)
